# ArXiv Enhanced Inference Pipeline
Inference on English ArXiv dataset with abstraction-aware evaluation metrics and LLM-as-Judge.

**Extension: Semantic Supervision on ArXiv** - Comparing with original paper on same dataset.

**Metrics:**
- **Abstraction Score**: Measures how much the summary differs from source (1 - n-gram overlap)
- **Compression Ratio**: Generated length / Source length
- **LLM Judge Score**: Llama evaluates quality on 1-5 scale across 4 dimensions

## 1. Setup

In [1]:
# Install deps
!pip install -q transformers datasets accelerate bitsandbytes sentence-transformers \
    spacy rouge_score bert_score langchain langchain-community langchain-huggingface \

!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 49.7 MB/s  0:00:00m0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [2]:
import os
import gc
import re
import torch
import json
import spacy
import numpy as np
from datetime import datetime
from tqdm.auto import tqdm
from collections import Counter
from datasets import load_dataset
from transformers import (
    AutoTokenizer, 
    AutoModelForTokenClassification, 
    AutoModelForCausalLM, 
    BitsAndBytesConfig, 
    pipeline
)
from langchain_huggingface import HuggingFacePipeline
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from rouge_score import rouge_scorer
from bert_score import score as bert_score
from huggingface_hub import login

# Load spacy for sentence segmentation
nlp = spacy.load("en_core_web_sm")

## 2. Configuration

In [3]:
# Auth
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except:
    HF_TOKEN = os.getenv("HF_TOKEN") or "YOUR_HF_TOKEN_HERE"

login(token=HF_TOKEN)

# ArXiv model configuration
SIGEXT_CONFIG = {
    "model_id": "LookUpMark/sigext-arxiv-en-1k-060t",
    "skip_samples": 10000,
    "threshold": 0.60
}

# Use 4-bit for best efficiency
QUANT_CONFIG = {
    "load_in_4bit": True,
    "bnb_4bit_compute_dtype": torch.float16,
    "bnb_4bit_quant_type": "nf4",
    "bnb_4bit_use_double_quant": True
}

GLOBAL_CONFIG = {
    "llm_model_id": "meta-llama/Llama-3.1-8B-Instruct",
    "num_test_samples": 100,
    "max_length": 2048,
    "output_dir": "./results_enhanced"
}

os.makedirs(GLOBAL_CONFIG["output_dir"], exist_ok=True)
print(f"Testing with {GLOBAL_CONFIG['num_test_samples']} samples")

Testing with 100 samples


## 3. Enhanced Prompts

In [4]:
# English ArXiv summarization prompt - Anti-Hallucination Version
SUMMARY_PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are a FAITHFUL scientific paper summarizer. Your summaries must contain ONLY facts from the source.

CRITICAL GROUNDING RULES:
1. Use ONLY information explicitly stated in the provided text.
2. DO NOT add external knowledge, dates, facts, or names not in the source.
3. If the source does not mention something, you must NOT mention it either.
4. When uncertain about a fact, OMIT it rather than guess.
5. DO NOT infer or extrapolate beyond what is written.

WRITING RULES:
1. Write EXACTLY ONE paragraph (100-150 words). NO titles, NO bullets.
2. SYNTHESIZE and REPHRASE - never copy sentences verbatim.
3. Focus on: main contribution, methodology, key results.
<|eot_id|><|start_header_id|>user<|end_header_id|>
SOURCE TEXT:
{source}

KEY CONCEPTS TO INTEGRATE (all from source above):
{keyphrases}

Write a FAITHFUL summary using ONLY facts from the source above. Do not add any external knowledge.<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""




# LLM-as-Judge prompt - SOURCE-AWARE Version (Full Source)
JUDGE_PROMPT = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
You are an expert summary evaluator. You have access to the full SOURCE document and the REFERENCE summary.
Evaluate if the GENERATED summary is faithful to the source content.

IMPORTANT: The generated summary may include information from the SOURCE that is NOT in the reference.
This is acceptable as long as all information is verifiable in the SOURCE.

Respond ONLY with valid JSON. Use double quotes. Max 15 words per reason.
<|eot_id|><|start_header_id|>user<|end_header_id|>

SOURCE DOCUMENT:
{source}

REFERENCE SUMMARY:
{reference}

GENERATED SUMMARY:
{generated}

---
Rate (1-5 each):
1. FAITHFULNESS: All facts verifiable in SOURCE? (1=fabricated, 5=all verified)
2. COMPLETENESS: Covers main points? (1=missing, 5=complete)
3. CONCISENESS: Fluid, not repetitive? (1=verbose, 5=concise)
4. ABSTRACTION: Rephrases vs copies? (1=verbatim, 5=novel phrasing)

{{"faithfulness": X, "faithfulness_reason": "...", "completeness": X, "completeness_reason": "...", "conciseness": X, "conciseness_reason": "...", "abstraction": X, "abstraction_reason": "..."}}
<|eot_id|><|start_header_id|>assistant<|end_header_id|>"""

## 4. Helper Functions

In [5]:
def get_test_data(skip_samples, num_samples):
    print(f"  Loading test data (skipping {skip_samples})...")
    dataset = load_dataset("ccdv/arxiv-summarization", split="train", streaming=True)
    dataset = dataset.skip(skip_samples)
    
    test_data = []
    for entry in dataset:
        source = entry['article']
        summary = entry['abstract']
        
        if len(source) < 500 or len(summary) < 50 or len(source) > 10000:
            continue
            
        test_data.append({"source": source, "reference": summary})
        
        if len(test_data) >= num_samples:
            break
    
    print(f"  Test data ready: {len(test_data)} samples")
    return test_data


def load_sigext_model(model_id):
    print(f"  Loading SigExt: {model_id}...")
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForTokenClassification.from_pretrained(model_id).to("cuda")
    return model, tokenizer


def load_llm(model_id, quant_config):
    quant_name = "4-bit" if "load_in_4bit" in quant_config else "8-bit"
    print(f"  Loading LLM ({quant_name}): {model_id}...")
    
    bnb_config = BitsAndBytesConfig(**quant_config)
    
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto"
    )
    tokenizer = AutoTokenizer.from_pretrained(model_id)
    tokenizer.pad_token = tokenizer.eos_token
    
    return model, tokenizer


def extract_salient_sentences(text, model, tokenizer, max_length):
    sentences = [sent.text.strip() for sent in nlp(text).sents if len(sent.text.strip()) > 20]
    
    if not sentences:
        return [], ""
    
    salient_sentences = []
    
    for sent in sentences:
        inputs = tokenizer(
            sent,
            return_tensors="pt",
            truncation=True,
            max_length=max_length
        ).to("cuda")
        
        with torch.no_grad():
            logits = model(**inputs).logits
        
        preds = torch.argmax(logits, dim=2)[0].tolist()
        
        valid_preds = preds[1:-1] if len(preds) > 2 else preds
        if valid_preds:
            salient_ratio = sum(valid_preds) / len(valid_preds)
            if salient_ratio > 0.5:
                salient_sentences.append(sent)
    
    keyphrases_text = "\n".join(f"- {s}" for s in salient_sentences)
    
    return salient_sentences, keyphrases_text


def preprocess_dataset(test_data, sigext_model, sigext_tokenizer, max_length):
    processed_data = []
    for item in tqdm(test_data, desc="    Extracting Salient Sentences"):
        salient_sents, keys_text = extract_salient_sentences(
            item['source'], sigext_model, sigext_tokenizer, max_length
        )
        processed_data.append({
            "source": item['source'],
            "reference": item['reference'],
            "salient_sentences": salient_sents,
            "keyphrases": keys_text
        })
    return processed_data


def clear_gpu_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

clear_gpu_memory()

## 5. New Abstraction Metrics

In [6]:
def get_ngrams(text, n=3):
    """Extract n-grams from text."""
    words = text.lower().split()
    return [tuple(words[i:i+n]) for i in range(len(words)-n+1)]


def compute_abstraction_score(source, generated, n=3):
    """
    Compute abstraction score: 1 - (n-gram overlap with source).
    Higher = more abstractive (less copying).
    """
    source_ngrams = set(get_ngrams(source, n))
    gen_ngrams = get_ngrams(generated, n)
    
    if not gen_ngrams:
        return 1.0  # Empty summary = no copying
    
    copied = sum(1 for ng in gen_ngrams if ng in source_ngrams)
    copy_ratio = copied / len(gen_ngrams)
    
    return 1.0 - copy_ratio


def compute_compression_ratio(source, generated):
    """Compute compression ratio (lower = more compressed)."""
    if len(source) == 0:
        return 1.0
    return len(generated) / len(source)


def compute_novel_ngrams(source, generated, n=2):
    """
    Compute percentage of n-grams in generated that are NOT in source.
    Higher = more novel content.
    """
    source_ngrams = set(get_ngrams(source, n))
    gen_ngrams = get_ngrams(generated, n)
    
    if not gen_ngrams:
        return 0.0
    
    novel = sum(1 for ng in gen_ngrams if ng not in source_ngrams)
    return novel / len(gen_ngrams)

## 6. LLM-as-Judge Evaluation

In [7]:
def parse_judge_response(text):
    """Robust JSON parsing for LLM judge responses."""
    import re
    import json
    
    KEYS = ['faithfulness', 'completeness', 'conciseness', 'abstraction']
    
    # Method 1: Try direct JSON parsing
    json_match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
    if json_match:
        json_str = json_match.group()
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            pass
    
    # Method 2: Clean common issues and retry
    if json_match:
        json_str = json_match.group()
        json_str = json_str.replace("'", '"')
        json_str = re.sub(r',\s*}', '}', json_str)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            pass
    
    # Method 3: Extract individual values with regex
    scores = {}
    for key in KEYS:
        match = re.search(rf'"{key}"\s*:\s*(\d)', text, re.IGNORECASE)
        if match:
            scores[key] = int(match.group(1))
        reason_match = re.search(rf'"{key}_reason"\s*:\s*"([^"]*)"', text)
        if reason_match:
            scores[f"{key}_reason"] = reason_match.group(1)
    
    return scores


def llm_judge_evaluate(source, generated, reference, judge_chain):
    """Use LLM to judge quality of generated summary against source."""
    ENGLISH_KEYS = ['faithfulness', 'completeness', 'conciseness', 'abstraction']
    DEFAULT_SCORES = {k: 3 for k in ENGLISH_KEYS}
    DEFAULT_SCORES.update({f"{k}_reason": "Unable to evaluate" for k in ENGLISH_KEYS})
    
    try:
        result = judge_chain.invoke({
            "source": source,
            "reference": reference,
            "generated": generated
        })
        
        result_text = result.split("assistant<|end_header_id|>")[-1].strip()
        scores = parse_judge_response(result_text)
        
        for key in ENGLISH_KEYS:
            if key not in scores:
                scores[key] = 3
            else:
                scores[key] = max(1, min(5, int(scores[key])))
            reason_key = f"{key}_reason"
            if reason_key not in scores:
                scores[reason_key] = ""
        
        return scores
            
    except Exception as e:
        print(f"      Judge error: {e}")
        return DEFAULT_SCORES.copy()


## 7. Enhanced Evaluation Function

In [8]:
def run_enhanced_evaluation(processed_data, summary_chain, judge_chain):
    
    scorer = rouge_scorer.RougeScorer(['rouge1'], use_stemmer=True)
    
    metrics = {
        # Traditional
        "bert": [], "rouge1": [], "kir": [],
        # Abstraction
        "abstraction": [], "compression": [], "novel_ngrams": [],
        # LLM Judge
        "judge_faithfulness": [], "judge_completeness": [], 
        "judge_conciseness": [], "judge_abstraction": [],
    }
    samples = []
    
    for item in tqdm(processed_data, desc="    Generating & Evaluating"):
        try:
            keys_text = item['keyphrases']
            salient_sents = item['salient_sentences']
            
            # Generate summary
            res = summary_chain.invoke({"source": item['source'], "keyphrases": keys_text})
            gen_summary = res.split("assistant<|end_header_id|>")[-1].strip()
            
            # === TRADITIONAL METRICS ===
            
            # ROUGE-1 and ROUGE-L
            rouge_scores = scorer.score(item['reference'], gen_summary)
            metrics["rouge1"].append(rouge_scores['rouge1'].fmeasure)
                        
            # BERTScore (English)
            _, _, F1 = bert_score([gen_summary], [item['reference']], lang="en", verbose=False)
            bert_sc = F1.mean().item()
            metrics["bert"].append(bert_sc)
            
            # KIR
            kir_score = 0.0
            if salient_sents:
                gen_lower = gen_summary.lower()
                hits = 0
                for sent in salient_sents:
                    words = [w.lower() for w in sent.split() if len(w) > 4]
                    if words:
                        word_hits = sum(1 for w in words if w in gen_lower)
                        if word_hits / len(words) > 0.3:
                            hits += 1
                kir_score = hits / len(salient_sents)
            metrics["kir"].append(kir_score)
            
            # === ABSTRACTION METRICS ===
            
            abstraction = compute_abstraction_score(item['source'], gen_summary)
            compression = compute_compression_ratio(item['source'], gen_summary)
            novel = compute_novel_ngrams(item['source'], gen_summary)
            
            metrics["abstraction"].append(abstraction)
            metrics["compression"].append(compression)
            metrics["novel_ngrams"].append(novel)
                        
            # === LLM JUDGE ===
            
            judge_scores = llm_judge_evaluate(
                item["source"], gen_summary, item["reference"], judge_chain
            )
            
            metrics["judge_faithfulness"].append(judge_scores["faithfulness"])
            metrics["judge_completeness"].append(judge_scores["completeness"])
            metrics["judge_conciseness"].append(judge_scores["conciseness"])
            metrics["judge_abstraction"].append(judge_scores["abstraction"])
            
            # Store sample details
            samples.append({
                "source": item['source'][:500] + "..." if len(item['source']) > 500 else item['source'],
                "reference": item['reference'],
                "salient_sentences": salient_sents,
                "generated_summary": gen_summary,
                "scores": {
                    "bert": float(bert_sc),
                    "rouge1": float(rouge_scores['rouge1'].fmeasure),
                    "kir": float(kir_score),
                    "abstraction": float(abstraction),
                    "compression": float(compression),
                    "novel_ngrams": float(novel),
                    
                    "judge": judge_scores
                }
            })
                
        except Exception as e:
            print(f"    Error: {e}")
            continue
    
    return metrics, samples


## 8. Main Evaluation Loop

In [9]:
print("="*60)
print("PHASE 1: LOADING MODELS")
print("="*60)

# Load SigExt
sigext_model, sigext_tokenizer = load_sigext_model(SIGEXT_CONFIG["model_id"])

# Load test data
test_data = get_test_data(
    SIGEXT_CONFIG["skip_samples"],
    GLOBAL_CONFIG["num_test_samples"]
)

# Extract salient sentences
processed_data = preprocess_dataset(
    test_data,
    sigext_model,
    sigext_tokenizer,
    GLOBAL_CONFIG["max_length"]
)

# Cleanup SigExt
del sigext_model, sigext_tokenizer
clear_gpu_memory()

print("\n" + "="*60)
print("PHASE 2: LOADING LLM")
print("="*60)

# Load LLM
llm_model, llm_tokenizer = load_llm(
    GLOBAL_CONFIG["llm_model_id"], 
    QUANT_CONFIG
)

# Create text generation pipeline
gen_pipe = pipeline(
    "text-generation",
    model=llm_model,
    tokenizer=llm_tokenizer,
    max_new_tokens=256,
    temperature=0.1
)

llm = HuggingFacePipeline(pipeline=gen_pipe)

# Create chains
summary_prompt = PromptTemplate(template=SUMMARY_PROMPT, input_variables=["source", "keyphrases"])
summary_chain = summary_prompt | llm | StrOutputParser()

judge_prompt = PromptTemplate(
    template=JUDGE_PROMPT, 
    input_variables=["source", "reference", "generated"]
)
judge_chain = judge_prompt | llm | StrOutputParser()

print("\n" + "="*60)
print("PHASE 3: RUNNING EVALUATION")
print("="*60)

metrics, samples = run_enhanced_evaluation(processed_data, summary_chain, judge_chain)

print("\n" + "="*60)
print("EVALUATION COMPLETE!")
print("="*60)

PHASE 1: LOADING MODELS
  Loading SigExt: LookUpMark/sigext-arxiv-en-1k-060t...


  Loading test data (skipping 10000)...
  Test data ready: 100 samples


    Extracting Salient Sentences:   0%|          | 0/100 [00:00<?, ?it/s]


PHASE 2: LOADING LLM
  Loading LLM (4-bit): meta-llama/Llama-3.1-8B-Instruct...


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]


PHASE 3: RUNNING EVALUATION


    Generating & Evaluating:   0%|          | 0/100 [00:00<?, ?it/s]


EVALUATION COMPLETE!


## 9. Results Summary

In [10]:
# Compute and display results
results = {
    "run_info": {
        "timestamp": datetime.now().isoformat(),
        "sigext_model": SIGEXT_CONFIG["model_id"],
        "llm_model": GLOBAL_CONFIG["llm_model_id"],
        "num_samples": len(samples),
        "prompt_type": "optimized_abstractive_v2"
    },
    "metrics": {}
}

# Aggregate metrics
for m in ["bert", "rouge1", "kir", "abstraction", "compression", "novel_ngrams"]:
    if metrics.get(m):
        results["metrics"][m] = {"mean": float(np.mean(metrics[m])), "std": float(np.std(metrics[m]))}

for m in ["judge_faithfulness", "judge_completeness", "judge_conciseness", "judge_abstraction"]:
    if metrics.get(m):
        results["metrics"][m] = {"mean": float(np.mean(metrics[m])), "std": float(np.std(metrics[m]))}

if metrics.get("judge_faithfulness"):
    overall = [sum(metrics[f"judge_{k}"][i] for k in ["faithfulness", "completeness", "conciseness", "abstraction"]) 
               for i in range(len(metrics["judge_faithfulness"]))]
    results["metrics"]["judge_overall"] = {"mean": float(np.mean(overall)) / 4, "std": float(np.std(overall)) / 4}

results["samples"] = samples

# Print summary
print("=" * 60)
print("RESULTS SUMMARY")
print("=" * 60)

m = results["metrics"]
if "bert" in m:
    print(f"\nTraditional Metrics:")
    print(f"  BERT Score:  {m['bert']['mean']:.4f} +/- {m['bert']['std']:.4f}")
    print(f"  ROUGE-1:     {m['rouge1']['mean']:.4f} +/- {m['rouge1']['std']:.4f}")
    print(f"  KIR:         {m['kir']['mean']:.2%}")

if "abstraction" in m:
    print(f"\nAbstraction Metrics:")
    print(f"  Abstraction: {m['abstraction']['mean']:.4f}")
    print(f"  Novel:       {m['novel_ngrams']['mean']:.2%}")
    print(f"  Compression: {m['compression']['mean']:.2%}")

if "judge_faithfulness" in m:
    print(f"\nLLM Judge (1-5):")
    print(f"  Faithfulness: {m['judge_faithfulness']['mean']:.2f}")
    print(f"  Completeness: {m['judge_completeness']['mean']:.2f}")
    print(f"  Conciseness:  {m['judge_conciseness']['mean']:.2f}")
    print(f"  Abstraction:  {m['judge_abstraction']['mean']:.2f}")
    print(f"  Overall:      {m['judge_overall']['mean']:.2f}")

# Save
output_file = os.path.join(GLOBAL_CONFIG["output_dir"], "results_enhanced.json")
with open(output_file, 'w') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print(f"\nSaved: {output_file}")


RESULTS SUMMARY

Traditional Metrics:
  BERT Score:  0.8194 +/- 0.0282
  ROUGE-1:     0.2954 +/- 0.1552
  KIR:         53.55%

Abstraction Metrics:
  Abstraction: 0.6145
  Novel:       45.78%
  Compression: 25.99%

LLM Judge (1-5):
  Faithfulness: 4.98
  Completeness: 4.94
  Conciseness:  4.77
  Abstraction:  4.91
  Overall:      4.90

Saved: ./results_enhanced/results_enhanced.json


## 10. Sample Analysis

In [11]:
# Sample outputs
print("=" * 60)
print("SAMPLE OUTPUTS")
print("=" * 60)

for idx in [0, len(samples)//2, len(samples)-1]:
    s = samples[idx]
    print(f"\n[Sample {idx}]")
    print(f"Reference: {s['reference'][:200]}...")
    print(f"Generated: {s['generated_summary'][:200]}...")
    sc = s['scores']
    print(f"Scores: BERT={sc['bert']:.2f} ROUGE={sc['rouge1']:.2f} KIR={sc['kir']:.2f} Abstr={sc['abstraction']:.2f}")


SAMPLE OUTPUTS

[Sample 0]
Reference: we have studied the fir/_mm _ spectrum of ir galaxies by combining iras photometry with new _ mm _ data on a complete southern iras galaxy sample . 
 the observed spectra and a dust model emphasize a ...
Generated: The authors investigated the dust content of spirals by observing the 1.25 mm continuum emission from a complete sample of IRAS galaxies with the SEST telescope. This allowed them to directly estimate...
Scores: BERT=0.84 ROUGE=0.43 KIR=0.34 Abstr=0.54

[Sample 50]
Reference: nicos , nightly control system , is a flexible tool for coordination of software development in large - scale projects . 
 it manages the multi - platform nightly builds based on the recent versions o...
Generated: Nicos is a nightly control system designed to facilitate collaborative software organization and management in high-energy physics experiments. It is an international project with hundreds of programm...
Scores: BERT=0.86 ROUGE=0.42 KIR=0.68 Abstr=0.59

[

## 11. Cleanup

In [12]:
# Cleanup
del llm_model, llm_tokenizer, gen_pipe, llm
clear_gpu_memory()

print(" Cleanup complete!")

 Cleanup complete!
